# Sessione 8 · App completata e demo

**Modulo 4 · La tua prima app di IA** · 2 sessioni · progetto reale · dall'idea al prodotto

> Ultima sessione del corso. Trasformiamo il prototipo in un'app curata, la pubblichiamo su internet perché chiunque la usi e la presentiamo. Chiudiamo con una mappa su dove proseguire.

**Al termine della sessione saprai:**
- Costruire un'interfaccia di **chat con memoria**
- Rafforzare la logica e la gestione degli errori
- **Pubblicare** l'app su internet (Streamlit Community Cloud)
- Presentare una demo e sapere cosa imparare dopo

*Risultato della sessione: hai un'app di IA funzionante, pubblicata e nel tuo portfolio.*

## 1. Da prototipo a prodotto: cosa manca?

Il prototipo della Sessione 7 funziona, ma come prodotto è ancora limitato. Ripassiamo cosa migliorare:

- **Memoria** — ora ogni domanda è indipendente; una buona chat **ricorda** la conversazione
- **Interfaccia di chat** — una casella e un pulsante bastano a poco; vogliamo vere bolle di chat
- **Scelta del fornitore** — dare la possibilità di scegliere tra Claude, OpenRouter e Ollama
- **Robustezza** — che non si rompa se fallisce la rete o manca la chiave (gestione degli errori)
- **Chiavi sicure** — non scrivere la chiave nel codice, soprattutto in vista della pubblicazione

Procediamo pezzo per pezzo e poi mettiamo tutto insieme.

## 2. Interfaccia di chat con memoria

Streamlit porta con sé tre strumenti che ci risolvono la chat quasi per intero:

- **`st.chat_message("user" / "assistant")`** — disegna la bolla di ogni messaggio
- **`st.chat_input(...)`** — la casella di scrittura fissa in basso, come in qualsiasi messaggistica
- **`st.session_state`** — la "memoria" dell'app tra un'interazione e l'altra. Qui salviamo la lista dei messaggi

L'idea della memoria è la stessa della Sessione 6: manteniamo una **lista di messaggi** e, a ogni turno, la mostriamo per intero e la inviamo completa al modello. Streamlit riesegue lo script a ogni interazione, per questo serve `session_state` per non perdere la cronologia.

## 3. Rafforzare la logica

Tre dettagli che trasformano il prototipo in qualcosa di serio:

- **Gestione degli errori** — avvolgiamo la chiamata all'LLM in `try/except`. Se qualcosa fallisce, mostriamo un messaggio cordiale invece di una schermata di errore
- **Cache del PDF** — con `@st.cache_data` leggiamo il PDF una sola volta, non a ogni messaggio (più veloce)
- **System prompt affinato** — insistiamo affinché risponda solo in base alle FAQ e riconosca quando non sa qualcosa, per evitare invenzioni

## 4. L'app finale, completa

Questa cella salva la versione definitiva in `chatbot_faq.py`. Riunisce tutto quanto sopra: lettura del PDF in cache, selezione del fornitore (i tre del corso), chiavi sicure, interfaccia di chat con memoria e gestione degli errori. Leggila con calma: riconoscerai ogni blocco di ciò che abbiamo fatto finora.

In [ ]:
%%writefile chatbot_faq.py
# -*- coding: utf-8 -*-
"""
Chatbot di Domande Frequenti (FAQ) per l'Assistenza Clienti.
Progetto finale del corso · Modulo 4.

Come eseguirlo:
    1) pip install -r requirements.txt
    2) streamlit run chatbot_faq.py

L'app legge un PDF di FAQ, lo passa come contesto a un LLM e risponde
ai dubbi del cliente basandosi SOLO su quell'informazione.

Supporta tre fornitori: Claude (Anthropic), OpenRouter (gratis) e Ollama (locale).
"""
import os
import streamlit as st
from pypdf import PdfReader

# --------------------------------------------------------------------------
# 1. CONFIGURAZIONE DI BASE DELLA PAGINA
# --------------------------------------------------------------------------
st.set_page_config(page_title="Assistente FAQ", page_icon="💬")
RUTA_FAQ = "faq_horta_verde.pdf"   # sostituiscilo con il tuo PDF


# --------------------------------------------------------------------------
# 2. LEGGERE IL PDF DELLE FAQ (in cache per non rileggerlo a ogni interazione)
# --------------------------------------------------------------------------
@st.cache_data
def cargar_faq(ruta):
    """Legge il PDF e restituisce tutto il suo testo come stringa."""
    lector = PdfReader(ruta)
    texto = ""
    for pagina in lector.pages:
        texto += pagina.extract_text() + "\n"
    return texto


# --------------------------------------------------------------------------
# 3. CHIAMARE L'LLM (tre possibili fornitori)
# --------------------------------------------------------------------------
def responder(historial, system, proveedor, clave):
    """Invia il system prompt + la cronologia al fornitore scelto e restituisce il testo."""
    if proveedor == "Claude (Anthropic)":
        import anthropic
        cliente = anthropic.Anthropic(api_key=clave)
        r = cliente.messages.create(
            model="claude-haiku-4-5",
            max_tokens=500,
            system=system,
            messages=historial,
        )
        return r.content[0].text

    elif proveedor == "OpenRouter (gratis)":
        from openai import OpenAI
        cliente = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=clave)
        mensajes = [{"role": "system", "content": system}] + historial
        r = cliente.chat.completions.create(
            model="meta-llama/llama-4-maverick:free",
            messages=mensajes,
            max_tokens=500,
        )
        return r.choices[0].message.content

    else:  # Ollama (locale)
        from ollama import chat
        mensajes = [{"role": "system", "content": system}] + historial
        return chat(model="llama3.2", messages=mensajes).message.content


# --------------------------------------------------------------------------
# 4. BARRA LATERALE: scegliere il fornitore e inserire la chiave
# --------------------------------------------------------------------------
st.sidebar.header("Configurazione")
proveedor = st.sidebar.selectbox(
    "Fornitore del modello",
    ["Claude (Anthropic)", "OpenRouter (gratis)", "Ollama (locale)"],
)

def leer_secreto(nombre):
    """Legge una chiave da st.secrets senza fallire se il file dei secrets non esiste."""
    try:
        return st.secrets[nombre]
    except Exception:
        return ""

# La chiave viene cercata prima nei 'secrets' (per la pubblicazione), poi nelle
# variabili d'ambiente e, se non c'è, viene chiesta nella barra laterale
clave = ""
if proveedor != "Ollama (locale)":
    nombre_secreto = "ANTHROPIC_API_KEY" if proveedor.startswith("Claude") else "OPENROUTER_API_KEY"
    clave = leer_secreto(nombre_secreto) or os.environ.get(nombre_secreto, "")
    if not clave:
        clave = st.sidebar.text_input("Chiave API", type="password")

if st.sidebar.button("🗑️ Riavvia conversazione"):
    st.session_state.mensajes = []
    st.rerun()


# --------------------------------------------------------------------------
# 5. ISTRUZIONI DELL'ASSISTENTE (system prompt)
# --------------------------------------------------------------------------
texto_faq = cargar_faq(RUTA_FAQ)

SYSTEM = f"""Sei l'assistente virtuale dell'assistenza clienti di Horta Verde, un negozio online.
Il tuo compito è rispondere ai dubbi dei clienti in modo cordiale, chiaro e breve, in italiano.

Regole importanti:
- Rispondi ESCLUSIVAMENTE con le informazioni delle Domande Frequenti riportate più sotto.
- Se la risposta non è nelle FAQ, dillo con onestà e invita a contattare l'assistenza
  clienti (hola@hortaverde.example o 900 123 456). Non inventare dati.
- Sii vicino e professionale. Non usare tecnicismi.

=== DOMANDE FREQUENTI ===
{texto_faq}
=== FINE DELLE FAQ ==="""


# --------------------------------------------------------------------------
# 6. INTERFACCIA DI CHAT
# --------------------------------------------------------------------------
st.title("💬 Assistente dell'assistenza clienti")
st.caption("Chiedimi di ordini, spedizioni, resi, pagamenti o del tuo account.")

# Inizializzare la cronologia della conversazione
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []

# Mostrare i messaggi precedenti
for m in st.session_state.mensajes:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

# Casella di input dell'utente
pregunta = st.chat_input("Scrivi la tua domanda...")
if pregunta:
    # Salvare e mostrare la domanda dell'utente
    st.session_state.mensajes.append({"role": "user", "content": pregunta})
    with st.chat_message("user"):
        st.markdown(pregunta)

    # Verificare che ci sia la chiave se il fornitore la richiede
    if proveedor != "Ollama (locale)" and not clave:
        with st.chat_message("assistant"):
            st.warning("Inserisci la tua chiave API nella barra laterale per poter rispondere.")
    else:
        with st.chat_message("assistant"):
            with st.spinner("Sto pensando..."):
                try:
                    respuesta = responder(st.session_state.mensajes, SYSTEM, proveedor, clave)
                except Exception as error:
                    respuesta = (f"Mi dispiace, si è verificato un problema tecnico "
                                 f"({type(error).__name__}). Riprova tra qualche secondo.")
            st.markdown(respuesta)
        st.session_state.mensajes.append({"role": "assistant", "content": respuesta})


Lanciala nel tuo terminale:

```bash
pip install -r requirements.txt
streamlit run chatbot_faq.py
```

Scegli il fornitore nella barra laterale, inserisci la tua chiave (oppure usa Ollama senza chiave) e conversa. Prova a fare una domanda, e poi una domanda di seguito che dipenda dalla precedente, per verificare che **ricorda**.

> Il file `requirements.txt` (te lo consegno con il materiale) elenca le librerie di cui l'app ha bisogno. È indispensabile per la pubblicazione del passo successivo.

## 5. Pubblicare: far vivere la tua app su internet

Finora l'app gira solo sul tuo computer. **Pubblicare** = metterla su un server perché chiunque la apra con un link. La via più semplice e gratuita è **Streamlit Community Cloud**.

I passi, in ordine:

1. **Carica il tuo progetto su GitHub** — crea un repository gratuito con tre file: `chatbot_faq.py`, `requirements.txt` e `faq_horta_verde.pdf`
2. **Vai su `share.streamlit.io`** con il tuo account GitHub
3. **Premi "New app"** e seleziona il tuo repository e il file `chatbot_faq.py`
4. **Configura la chiave in modo sicuro** — in *Advanced settings → Secrets*, incolla la tua chiave così:
   ```toml
   ANTHROPIC_API_KEY = "sk-ant-..."
   ```
   L'app la legge con `st.secrets`, senza che la chiave finisca mai nel codice pubblico
5. **Premi "Deploy"** — in uno o due minuti avrai un link pubblico da condividere

> ⚠️ **Non caricare mai la tua chiave su GitHub.** Per questo usiamo i *Secrets*: il codice è pubblico, la chiave no. Se una chiave viene esposta, revocala subito nella console del fornitore.

## 6. Demo e revisione del progetto

Prima di presentare, ripassa questa **lista di controllo** (provalo tutto dal vivo):

- Risponde bene alle domande che **sono** nelle FAQ (spedizioni, resi, pagamenti)
- Di fronte a una domanda che **non c'è**, riconosce di non saperlo e rimanda al supporto (non inventa)
- **Ricorda** il contesto in una domanda di seguito
- Non si rompe se sbagli chiave o cade la connessione (mostra un messaggio cordiale)
- Il pulsante di **riavvio** pulisce la conversazione

**Scaletta suggerita per la demo (3 minuti):**
1. Spiega il problema in una frase: "assistenza clienti 24/7 senza inventare nulla"
2. Mostra l'architettura: PDF → LLM → web
3. Fai due domande dal vivo: una semplice e una di seguito
4. Provoca di proposito una domanda fuori dalle FAQ per mostrare che **non allucina**
5. Chiudi con il link pubblico pubblicato

## 7. Prossimi passi: cosa imparare dopo

Hai chiuso il ciclo completo: da non sapere cos'è l'IA a pubblicare un'app. Se vuoi continuare a crescere, queste sono le direzioni naturali:

- **RAG (ricerca + generazione)** — per documenti grandi che non entrano nel contesto. Il testo viene spezzettato, convertito in vettori (*embedding*) e si recupera solo ciò che è rilevante per ogni domanda. È il livello successivo di questo stesso progetto
- **Valutazione** — misurare in modo sistematico se le risposte sono corrette, con un insieme di domande di test
- **Più fonti di dati** — collegare più PDF, un sito web o un database
- **Registrazione e miglioramento continuo** — salvare le domande reali per individuare lacune nelle FAQ
- **Altre interfacce** — integrarlo in WhatsApp, nel sito dell'azienda o in un widget

> Il mio consiglio: **non accumulare teoria, costruisci**. Prendi un problema reale del tuo ambito e ripeti questo ciclo — dati, modello, interfaccia — con quello. Si impara consegnando progetti, non leggendo su di essi.

## 8. Riepilogo del corso

Guarda indietro un momento. In otto sessioni hai percorso:

- **Modulo 1** — cos'è l'IA e come imparano le macchine, senza una sola formula
- **Modulo 2** — Python da zero e il tuo primo modello con scikit-learn
- **Modulo 3** — i grandi modelli di linguaggio, *prompt engineering* e le loro API
- **Modulo 4** — un progetto reale dall'inizio alla fine: un chatbot di FAQ pubblicato

Sei passato da **zero** ad avere un'app di IA funzionante su internet. Questo salto è esattamente ciò che distingue chi "ha sentito parlare di IA" da chi **sa applicarla**.

👉 **Pubblica il tuo progetto e condividilo.** È il miglior biglietto da visita di ciò che ormai sai fare.

Congratulazioni per aver completato il corso!